In [1]:
# 기본
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 경고 뜨지 않게 설정
import warnings
warnings.filterwarnings('ignore')

# 그래프 설정
sns.set()

# 그래프 기본 설정
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 데이터 전처리 알고리즘
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

# 학습용과 검증용으로 나누는 함수
from sklearn.model_selection import train_test_split

# 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold

# 평가함수
# 분류용
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

# 회귀용
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

# 모델의 최적의 하이퍼 파라미터를 찾기 위한 도구
from sklearn.model_selection import GridSearchCV

# 머신러닝 알고리즘 - 분류
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier

# 머신러닝 알고리즘 - 회귀
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import VotingRegressor


# 학습 모델 저장을 위한 라이브러리
import pickle

### 프로젝트 셋팅

In [2]:
# 학습이 완료된 모델을 저장할 파일 이름
best_model_path = 'model/best_model_final13.dat'
# 교차검증 횟수
cv_count = 10
# 교차 검증
kfold = KFold(n_splits=cv_count, shuffle=True, random_state=1)
# 평가 결과를 담을 리스트
# 필요하다면 다른 것도 만들어주세요
f1_score_list = []
# 학습 모델 이름
model_name_list = []

### 데이터 준비
- 데이터를 읽어오고 필요한 전처리까지 다 한다음 입력데이터는 train_X, 결과데이터는 train_y라는 변수에 담아서 준비해주세요

In [3]:
# 데이터를 읽어온다.
train_df = pd.read_csv('(E_NotE)_train.csv')
test_df = pd.read_csv('(E_NotE)_test.csv')

display(train_df)
display(test_df)

,Group,이용금액_R3M_신용체크,이용카드수_신용체크,이용카드수_신용,_1순위카드이용건수,청구금액_R6M,청구서발송여부_B0,할인건수_R3M,이용금액_오프라인_R6M,이용건수_오프라인_B0M,...,이용후경과월_신용,정상청구원금_B5M,연속유실적개월수_기본_24M_카드,이용개월수_전체_R6M,정상입금원금_B0M,이용금액대,방문횟수_앱_R6M,방문일수_PC_R6M,인입횟수_ARS_R6M,이용메뉴건수_ARS_R6M
0,Not_E,196,1,1,26,88693,1,1회 이상,11097,7,...,0,14958,13,6,6335,01.100만원+,1회 이상,1회 이상,10회 이상,10회 이상
1,E,13475,1,1,46,16861,1,1회 이상,18638,15,...,0,3367,12,6,5198,03.30만원+,1회 이상,1회 이상,1회 이상,1회 이상
2,Not_E,23988,1,1,28,165221,1,1회 이상,29192,12,...,0,23963,8,5,12564,01.100만원+,30회 이상,10회 이상,1회 이상,1회 이상
3,Not_E,3904,1,1,1,127371,1,1회 이상,18056,8,...,0,19614,5,6,7639,01.100만원+,1회 이상,1회 이상,10회 이상,10회 이상
4,E,1190,1,0,-2,155,0,1회 이상,787,0,...,6,0,0,1,0,09.미사용,1회 이상,1회 이상,1회 이상,1회 이상
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2399995,E,10755,1,0,3,0,0,1회 이상,0,0,...,8,0,0,0,0,09.미사용,1회 이상,1회 이상,1회 이상,1회 이상
2399996,Not_E,27636,1,1,38,99849,1,1회 이상,60373,7,...,0,23742,17,6,9705,01.100만원+,1회 이상,1회 이상,1회 이상,1회 이상
2399997,Not_E,23187,1,1,33,41073,1,1회 이상,32036,13,...,1,4125,24,6,5346,02.50만원+,1회 이상,1회 이상,1회 이상,1회 이상
2399998,E,0,0,0,-2,0,0,1회 이상,0,0,...,12,507,0,0,0,09.미사용,1회 이상,1회 이상,1회 이상,1회 이상


,이용금액대,할인건수_R3M,방문횟수_앱_R6M,방문일수_PC_R6M,인입횟수_ARS_R6M,이용메뉴건수_ARS_R6M,정상청구원금_B5M,이용금액_R3M_신용체크,이용개월수_신용_R12M,연속유실적개월수_기본_24M_카드,...,_2순위쇼핑업종_이용금액,이용카드수_신용체크,_3순위쇼핑업종_이용금액,정상입금원금_B0M,_3순위업종_이용금액,청구서발송여부_B0,이용후경과월_신용,_2순위업종_이용금액,이용카드수_신용,_1순위카드이용건수
0,02.50만원+,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,3919,21458,10,5,...,854,2,643,3680,1880,1,0,2713,2,51
1,02.50만원+,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,5723,18681,9,8,...,1154,2,1033,8726,1278,1,0,1321,1,40
2,01.100만원+,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,11267,40758,12,24,...,1178,2,1094,11297,2680,1,1,7271,2,154
3,04.10만원+,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,0,5255,9,5,...,804,1,634,1375,605,1,0,2043,1,105
4,03.30만원+,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1347,16148,12,20,...,774,3,760,3951,1054,1,0,1364,2,52
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
599995,09.미사용,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,0,0,0,0,...,0,0,0,0,0,0,12,0,0,-2
599996,05.10만원-,1회 이상,10회 이상,1회 이상,1회 이상,1회 이상,992,3110,10,8,...,0,1,0,205,0,1,0,0,1,4
599997,09.미사용,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,0,0,0,0,...,0,0,0,0,0,0,12,0,0,6
599998,01.100만원+,1회 이상,40회 이상,1회 이상,1회 이상,1회 이상,27335,173263,12,24,...,2208,6,1965,0,4612,1,0,16296,4,185


In [6]:
# 데이터 프레임을 합친다.
all_df = pd.concat([train_df, test_df])
all_df.reset_index(inplace=True, drop=True)
all_df

,Group,이용금액_R3M_신용체크,이용카드수_신용체크,이용카드수_신용,_1순위카드이용건수,청구금액_R6M,청구서발송여부_B0,할인건수_R3M,이용금액_오프라인_R6M,이용건수_오프라인_B0M,...,이용후경과월_신용,정상청구원금_B5M,연속유실적개월수_기본_24M_카드,이용개월수_전체_R6M,정상입금원금_B0M,이용금액대,방문횟수_앱_R6M,방문일수_PC_R6M,인입횟수_ARS_R6M,이용메뉴건수_ARS_R6M
0,Not_E,196,1,1,26,88693,1,1회 이상,11097,7,...,0,14958,13,6,6335,01.100만원+,1회 이상,1회 이상,10회 이상,10회 이상
1,E,13475,1,1,46,16861,1,1회 이상,18638,15,...,0,3367,12,6,5198,03.30만원+,1회 이상,1회 이상,1회 이상,1회 이상
2,Not_E,23988,1,1,28,165221,1,1회 이상,29192,12,...,0,23963,8,5,12564,01.100만원+,30회 이상,10회 이상,1회 이상,1회 이상
3,Not_E,3904,1,1,1,127371,1,1회 이상,18056,8,...,0,19614,5,6,7639,01.100만원+,1회 이상,1회 이상,10회 이상,10회 이상
4,E,1190,1,0,-2,155,0,1회 이상,787,0,...,6,0,0,1,0,09.미사용,1회 이상,1회 이상,1회 이상,1회 이상
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2999995,NaN,0,0,0,-2,0,0,1회 이상,0,0,...,12,0,0,0,0,09.미사용,1회 이상,1회 이상,1회 이상,1회 이상
2999996,NaN,3110,1,1,4,2237,1,1회 이상,2331,5,...,0,992,8,6,205,05.10만원-,10회 이상,1회 이상,1회 이상,1회 이상
2999997,NaN,0,0,0,6,0,0,1회 이상,0,0,...,12,0,0,0,0,09.미사용,1회 이상,1회 이상,1회 이상,1회 이상
2999998,NaN,173263,6,4,185,108420,1,1회 이상,89973,66,...,0,27335,24,6,0,01.100만원+,40회 이상,1회 이상,1회 이상,1회 이상


In [8]:
# 결과 데이터는 제거한다.
all_df.drop('Group', axis=1, inplace=True)
all_df

,이용금액_R3M_신용체크,이용카드수_신용체크,이용카드수_신용,_1순위카드이용건수,청구금액_R6M,청구서발송여부_B0,할인건수_R3M,이용금액_오프라인_R6M,이용건수_오프라인_B0M,_2순위쇼핑업종_이용금액,...,이용후경과월_신용,정상청구원금_B5M,연속유실적개월수_기본_24M_카드,이용개월수_전체_R6M,정상입금원금_B0M,이용금액대,방문횟수_앱_R6M,방문일수_PC_R6M,인입횟수_ARS_R6M,이용메뉴건수_ARS_R6M
0,196,1,1,26,88693,1,1회 이상,11097,7,0,...,0,14958,13,6,6335,01.100만원+,1회 이상,1회 이상,10회 이상,10회 이상
1,13475,1,1,46,16861,1,1회 이상,18638,15,435,...,0,3367,12,6,5198,03.30만원+,1회 이상,1회 이상,1회 이상,1회 이상
2,23988,1,1,28,165221,1,1회 이상,29192,12,1038,...,0,23963,8,5,12564,01.100만원+,30회 이상,10회 이상,1회 이상,1회 이상
3,3904,1,1,1,127371,1,1회 이상,18056,8,487,...,0,19614,5,6,7639,01.100만원+,1회 이상,1회 이상,10회 이상,10회 이상
4,1190,1,0,-2,155,0,1회 이상,787,0,0,...,6,0,0,1,0,09.미사용,1회 이상,1회 이상,1회 이상,1회 이상
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2999995,0,0,0,-2,0,0,1회 이상,0,0,0,...,12,0,0,0,0,09.미사용,1회 이상,1회 이상,1회 이상,1회 이상
2999996,3110,1,1,4,2237,1,1회 이상,2331,5,0,...,0,992,8,6,205,05.10만원-,10회 이상,1회 이상,1회 이상,1회 이상
2999997,0,0,0,6,0,0,1회 이상,0,0,0,...,12,0,0,0,0,09.미사용,1회 이상,1회 이상,1회 이상,1회 이상
2999998,173263,6,4,185,108420,1,1회 이상,89973,66,2208,...,0,27335,24,6,0,01.100만원+,40회 이상,1회 이상,1회 이상,1회 이상


In [9]:
all_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000000 entries, 0 to 2999999
Data columns (total 26 columns):
 #   Column              Dtype 
---  ------              ----- 
 0   이용금액_R3M_신용체크       int64 
 1   이용카드수_신용체크          int64 
 2   이용카드수_신용            int64 
 3   _1순위카드이용건수          int64 
 4   청구금액_R6M            int64 
 5   청구서발송여부_B0          int64 
 6   할인건수_R3M            object
 7   이용금액_오프라인_R6M       int64 
 8   이용건수_오프라인_B0M       int64 
 9   _2순위쇼핑업종_이용금액       int64 
 10  _3순위쇼핑업종_이용금액       int64 
 11  _3순위업종_이용금액         int64 
 12  _2순위업종_이용금액         int64 
 13  이용개월수_신용_R12M       int64 
 14  이용금액_일시불_B0M        int64 
 15  이용건수_신용_R12M        int64 
 16  이용후경과월_신용           int64 
 17  정상청구원금_B5M          int64 
 18  연속유실적개월수_기본_24M_카드  int64 
 19  이용개월수_전체_R6M        int64 
 20  정상입금원금_B0M          int64 
 21  이용금액대               object
 22  방문횟수_앱_R6M          object
 23  방문일수_PC_R6M         object
 24  인입횟수_ARS_R6M        object
 25  이용메뉴건수_ARS_R6M    

In [10]:
# LabelEncoder 학습
Encoder1 = LabelEncoder()
Encoder2 = LabelEncoder()
Encoder3 = LabelEncoder()
Encoder4 = LabelEncoder()
Encoder5 = LabelEncoder()
Encoder6 = LabelEncoder()

Encoder1.fit(all_df['할인건수_R3M'])
Encoder2.fit(all_df['이용금액대'])
Encoder3.fit(all_df['방문횟수_앱_R6M'])
Encoder4.fit(all_df['방문일수_PC_R6M'])
Encoder5.fit(all_df['인입횟수_ARS_R6M'])
Encoder6.fit(all_df['이용메뉴건수_ARS_R6M'])

LabelEncoder()

In [11]:
all_df['할인건수_R3M'] = Encoder1.transform(all_df['할인건수_R3M'])
all_df['이용금액대'] = Encoder2.transform(all_df['이용금액대'])
all_df['방문횟수_앱_R6M'] = Encoder3.transform(all_df['방문횟수_앱_R6M'])
all_df['방문일수_PC_R6M'] = Encoder4.transform(all_df['방문일수_PC_R6M'])
all_df['인입횟수_ARS_R6M'] = Encoder5.transform(all_df['인입횟수_ARS_R6M'])
all_df['이용메뉴건수_ARS_R6M'] = Encoder6.transform(all_df['이용메뉴건수_ARS_R6M'])

In [12]:
# Scaler 학습
scalerX = StandardScaler()
scalerX.fit(all_df)

,copy,True
,with_mean,True
,with_std,True


In [13]:
train_df['할인건수_R3M'] = Encoder1.transform(train_df['할인건수_R3M'])
train_df['이용금액대'] = Encoder2.transform(train_df['이용금액대'])
train_df['방문횟수_앱_R6M'] = Encoder3.transform(train_df['방문횟수_앱_R6M'])
train_df['방문일수_PC_R6M'] = Encoder4.transform(train_df['방문일수_PC_R6M'])
train_df['인입횟수_ARS_R6M'] = Encoder5.transform(train_df['인입횟수_ARS_R6M'])
train_df['이용메뉴건수_ARS_R6M'] = Encoder6.transform(train_df['이용메뉴건수_ARS_R6M'])

In [14]:
# 입력과 결과로 나눈다.
X = train_df.drop('Group', axis=1)
y = train_df['Group']

In [15]:
# 표준화
X2 = scalerX.transform(X)
X2

array([[-0.71919385, -0.33371433, -0.19290226, ...,  0.0641234 ,
        -5.69855686, -4.44973122],
       [-0.14848993, -0.33371433, -0.19290226, ...,  0.0641234 ,
         0.17548303,  0.08322662],
       [ 0.30333704, -0.33371433, -0.19290226, ..., -3.71384184,
         0.17548303,  0.08322662],
       ...,
       [ 0.26891172, -0.33371433, -0.19290226, ...,  0.0641234 ,
         0.17548303,  0.08322662],
       [-0.72761753, -1.33380024, -1.30411597, ...,  0.0641234 ,
         0.17548303,  0.08322662],
       [ 0.19481777,  0.66637157,  0.91831146, ...,  0.0641234 ,
         0.17548303,  0.08322662]])

In [25]:
scaler_columns = X.columns.tolist()
scaler_columns

['이용금액_R3M_신용체크',
 '이용카드수_신용체크',
 '이용카드수_신용',
 '_1순위카드이용건수',
 '청구금액_R6M',
 '청구서발송여부_B0',
 '할인건수_R3M',
 '이용금액_오프라인_R6M',
 '이용건수_오프라인_B0M',
 '_2순위쇼핑업종_이용금액',
 '_3순위쇼핑업종_이용금액',
 '_3순위업종_이용금액',
 '_2순위업종_이용금액',
 '이용개월수_신용_R12M',
 '이용금액_일시불_B0M',
 '이용건수_신용_R12M',
 '이용후경과월_신용',
 '정상청구원금_B5M',
 '연속유실적개월수_기본_24M_카드',
 '이용개월수_전체_R6M',
 '정상입금원금_B0M',
 '이용금액대',
 '방문횟수_앱_R6M',
 '방문일수_PC_R6M',
 '인입횟수_ARS_R6M',
 '이용메뉴건수_ARS_R6M']

In [16]:
train_X = X2
train_y = y

In [17]:
le = LabelEncoder()
train_y = le.fit_transform(train_y)

### 기본 모델 사용하기
- 기본 모델 중에 만족하는 것을 찾았다면 하이퍼 파라미터 튜닝 과정은 생략하세요

In [18]:
model5 = LGBMClassifier(device='cpu', objective='multiclass', num_class=5, verbose=-1)

kfold = KFold(n_splits=5, shuffle=True, random_state=1)
r1 = cross_val_score(model5, train_X, train_y, scoring='f1_micro', cv=kfold)
print(f'평균 f1 Score : {r1.mean()}')

f1_score_list.append(r1.mean())
model_name_list.append("LGBMClassifier")

평균 f1 Score : 0.9060295833333333


In [19]:
# CPU 기반 XGBoost 모델
model6 = XGBClassifier(
    n_jobs=-1,
    verbosity=0,
    use_label_encoder=False,
    eval_metric='mlogloss'
)

# 교차 검증
kfold = KFold(n_splits=10, shuffle=True, random_state=1)

# f1_weighted 사용
r2 = cross_val_score(model6, train_X, train_y, scoring='f1_micro', cv=kfold)
print(f'평균 f1 Score : {r2.mean():.4f}')

f1_score_list.append(r2.mean())
model_name_list.append("XGBClassifier")

평균 f1 Score : 0.9074


In [20]:
df = pd.DataFrame({
    'Model': model_name_list,
    'f1 score': f1_score_list
})
df = df.dropna()

In [21]:
df

,Model,f1 score
0,LGBMClassifier,0.906030
1,XGBClassifier,0.907386


In [22]:
final_model=model6.fit(train_X, train_y)

In [27]:
with open(best_model_path, 'wb') as fp:
    pickle.dump(model6, fp)
    pickle.dump(scalerX, fp)
    pickle.dump(scaler_columns, fp)
    pickle.dump(Encoder1, fp)
    pickle.dump(Encoder2, fp)
    pickle.dump(Encoder3, fp)
    pickle.dump(Encoder4, fp)
    pickle.dump(Encoder5, fp)
    pickle.dump(Encoder6, fp)
    pickle.dump(le, fp)  # ← LabelEncoder 객체 추가

print('저장완료')

저장완료
